# PATRÓN B — Ingesta incremental desde una API pública REST (JSON → Bronze)
* Fuente: Open-Meteo, API meteorológica gratuita, sin API key.
* URL: https://open-meteo.com/  (uso no comercial)

# Librerias base

In [18]:
import os
import requests
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from delta.tables import DeltaTable
from datetime import date, timedelta

StatementMeta(, 5ec9a63f-4330-45c0-8ee2-5e5140b3f8cf, 20, Finished, Available, Finished, False)

# Variables base

In [19]:
# ruta local del Lakehouse dentro del notebook
LAKEHOUSE_FILES = "/lakehouse/default/Files"
NAME_LH = "LH_BRONCE_ADVANCED"
NAME_SCHEMA = "OPENMATEO"
TABLA_BRONZE = f"{NAME_LH}.{NAME_SCHEMA}.clima_ciudades"

StatementMeta(, 5ec9a63f-4330-45c0-8ee2-5e5140b3f8cf, 21, Finished, Available, Finished, False)

# Definir ciudades y squema tabla

In [20]:
ciudades = [
    {"nombre": "Santiago",      "lat": -33.45, "lon": -70.67},
    {"nombre": "Buenos Aires",  "lat": -34.61, "lon": -58.38},
    {"nombre": "Ciudad de México", "lat": 19.43, "lon": -99.13},
]
 
squema_clima = StructType([
    StructField("ciudad", StringType()),
    StructField("hora", StringType()),
    StructField("temperatura_c", DoubleType()),
    StructField("ingestion_date", StringType()),
])

StatementMeta(, 5ec9a63f-4330-45c0-8ee2-5e5140b3f8cf, 22, Finished, Available, Finished, False)

# Consultar API

In [26]:
filas = []
# formato yyyy-MM-dd
hoy = date.today().isoformat()
#delta = (date.today() - timedelta(days=1)).isoformat()
#hoy = delta
 
for ciudad in ciudades:
    params = {
        "latitude": ciudad["lat"],
        "longitude": ciudad["lon"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()
 
    horas = payload["hourly"]["time"]
    temps = payload["hourly"]["temperature_2m"]
    for hora, temp in zip(horas, temps):
        filas.append((ciudad["nombre"], hora, float(temp), hoy))

df_clima = spark.createDataFrame(filas, schema=squema_clima)

StatementMeta(, 5ec9a63f-4330-45c0-8ee2-5e5140b3f8cf, 28, Finished, Available, Finished, False)

In [27]:
display(df_clima.limit(10))

StatementMeta(, 5ec9a63f-4330-45c0-8ee2-5e5140b3f8cf, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e2011f9e-42dd-4fb4-a6ea-84b609dcb57a)

# Ingesta incremental idempotente: MERGE en vez de append
* Permite re-ejecutar el notebook el mismo día (por ejemplo tras un fallo) sin duplicar filas.
* Aplicado en cargas programadas para un proceso que se ejecute mas de una vez al día.
* Merge: Un registro se considera existente si tiene la misma ciudad, la misma hora y la misma fecha de ingesta.

In [28]:
if spark.catalog.tableExists(TABLA_BRONZE):
    tabla_destino = DeltaTable.forName(spark, TABLA_BRONZE)
    (
        tabla_destino.alias("dest")
        .merge(
            df_clima.alias("src"),
            "dest.ciudad = src.ciudad AND dest.hora = src.hora AND dest.ingestion_date = src.ingestion_date",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_clima.write.format("delta").partitionBy("ingestion_date").saveAsTable(TABLA_BRONZE)
 
print(f"Ingesta de clima completada para {len(ciudades)} ciudades — {len(filas)} filas horarias.")

StatementMeta(, 5ec9a63f-4330-45c0-8ee2-5e5140b3f8cf, 30, Finished, Available, Finished, False)

Ingesta de clima completada para 3 ciudades — 72 filas horarias.


### Ejemplo de `MERGE`

#### Tabla destino

| ciudad   | hora  | ingestion_date |
|----------|-------|----------------|
| Santiago | 13:00 | 2026-09-02     |
| Santiago | 14:00 | 2026-09-02     |

#### `df_clima` — datos nuevos

| ciudad   | hora  | ingestion_date |
|----------|-------|----------------|
| Santiago | 14:00 | 2026-09-02     |
| Santiago | 15:00 | 2026-09-02     |

---

### ¿Qué ocurre durante el `MERGE`?

El registro de las **14:00**:

```text
MATCH
```

→ Ya existe en la tabla destino, por lo tanto **no hace nada**.

El registro de las **15:00**:

```text
NOT MATCHED
```

→ No existe en la tabla destino, por lo tanto **lo inserta**.

---

### Resultado final

| ciudad   | hora  | ingestion_date |
|----------|-------|----------------|
| Santiago | 13:00 | 2026-09-02     |
| Santiago | 14:00 | 2026-09-02     |
| Santiago | 15:00 | 2026-09-02     |

---

### ¿Cómo determina si el registro existe?

El `MERGE` compara los registros utilizando la siguiente condición:

```text
dest.ciudad = src.ciudad
AND
dest.hora = src.hora
AND
dest.ingestion_date = src.ingestion_date
```

Por lo tanto, la combinación:

**ciudad + hora + ingestion_date**

determina si el registro ya existe en la tabla destino.

### Regla del `MERGE`

```text
Si existe la combinación
ciudad + hora + ingestion_date
        ↓
      MATCH
        ↓
   No hace nada

Si NO existe la combinación
ciudad + hora + ingestion_date
        ↓
   NOT MATCHED
        ↓
      INSERT
```
